In [1]:
// Añadimos Apache Spark Core y SQL para Scala 2.13
// El sufijo _2.13 es gestionado automáticamente por $ivy cuando usamos %%
import $ivy.`org.apache.spark::spark-core:4.1.1`
import $ivy.`org.apache.spark::spark-sql:4.1.1`

println("✅ Dependencias de Spark cargadas correctamente")

✅ Dependencias de Spark cargadas correctamente


import $ivy.$
import $ivy.$

In [2]:
import org.apache.log4j.{Level, Logger}

// Silenciamos los logs de Spark para que el output sea legible
Logger.getLogger("org").setLevel(Level.ERROR)
Logger.getLogger("akka").setLevel(Level.ERROR)

println("✅ Logs de Spark configurados (solo se mostrarán errores)")

✅ Logs de Spark configurados (solo se mostrarán errores)


import org.apache.log4j.{Level, Logger}

In [3]:
import org.apache.spark.sql.SparkSession

// Creamos la SparkSession en modo local
// "local[*]" significa: usa todos los núcleos de CPU disponibles
val spark = SparkSession.builder()
  .appName("IntroSpark")   // nombre visible en el Spark UI
  .master("local[*]")                    // modo local, todos los núcleos
  .config("spark.ui.showConsoleProgress", "false")  // sin barras de progreso
  .getOrCreate()

// Obtenemos el SparkContext desde la sesión
val sc = spark.sparkContext

println(s"✅ SparkSession creada correctamente")
println(s"   Versión de Spark: ${spark.version}")
println(s"   Nombre de la app: ${sc.appName}")
println(s"   Master:           ${sc.master}")
println(s"   Spark UI:         http://localhost:4040")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/23 12:09:05 INFO SparkContext: Running Spark version 4.1.1
26/04/23 12:09:05 INFO SparkContext: OS info Windows 11, 10.0, amd64
26/04/23 12:09:05 INFO SparkContext: Java version 17.0.12+8-LTS-286
26/04/23 12:09:05 INFO ResourceUtils: ==============================================================
26/04/23 12:09:05 INFO ResourceUtils: No custom resources configured for spark.driver.
26/04/23 12:09:05 INFO ResourceUtils: ==============================================================
26/04/23 12:09:05 INFO SparkContext: Submitted application: IntroSpark
26/04/23 12:09:05 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/04/23 12:09:05 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/04/23 12:09:05 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/04/23 12:09:05 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/04/23 12:09:05 INFO SecurityManager: Changing view acls to

2026-04-23T10:09:06.276078100Z scala-kernel-interpreter-1 ERROR An exception occurred processing Appender console org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:164)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:133)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:124)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:88)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:714)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:672)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:648)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:584)
	at org.apache.logging.log4j.

import org.apache.spark.sql.SparkSession
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@66490271
sc: org.apache.spark.SparkContext = org.apache.spark.SparkContext@74184c1

2. Transformaciones elemento a elemento

2.1 map — transformar cada elemento

In [4]:
// Ejemplo: elevar al cuadrado una lista de números
val numeros = sc.parallelize(List(1, 2, 3, 4, 5))
val cuadrados = numeros.map(n => n * n)
cuadrados.collect()
// Array(1, 4, 9, 16, 25)

numeros: org.apache.spark.rdd.RDD[Int] = ParallelCollectionRDD[0] at parallelize at cmd4.sc:2
cuadrados: org.apache.spark.rdd.RDD[Int] = MapPartitionsRDD[1] at map at cmd4.sc:3
res4_2: Array[Int] = Array(1, 4, 9, 16, 25)

In [14]:
// Ejemplo: convertir a mayúsculas
val palabras = sc.parallelize(List("scala", "spark", "big data"))
val mayusculas = palabras.map(p => p.toUpperCase)
mayusculas.collect()
// Array("SCALA", "SPARK", "BIG DATA")
println(mayusculas.collect().mkString(", "))

SCALA, SPARK, BIG DATA


palabras: org.apache.spark.rdd.RDD[String] = ParallelCollectionRDD[7] at parallelize at cmd14.sc:2
mayusculas: org.apache.spark.rdd.RDD[String] = MapPartitionsRDD[8] at map at cmd14.sc:3
res14_2: Array[String] = Array("SCALA", "SPARK", "BIG DATA")

2.2 flatMap — transformar y aplanar
Como map, pero cada elemento puede producir cero, uno o varios elementos de salida. El resultado se "aplana" en un único RDD

In [15]:
val frases = sc.parallelize(List("hola mundo", "spark es rápido"))

// Con map obtenemos arrays dentro del RDD:
frases.map(f => f.split(" ")).collect()
// Array(Array("hola","mundo"), Array("spark","es","rápido"))

// Con flatMap obtenemos palabras individuales:
val palabras = frases.flatMap(f => f.split(" "))
palabras.collect()
// Array("hola", "mundo", "spark", "es", "rápido")
println(palabras.collect().mkString(", "))

hola, mundo, spark, es, rápido


frases: org.apache.spark.rdd.RDD[String] = ParallelCollectionRDD[9] at parallelize at cmd15.sc:1
res15_1: Array[Array[String]] = Array(
  Array("hola", "mundo"),
  Array("spark", "es", "rápido")
)
palabras: org.apache.spark.rdd.RDD[String] = MapPartitionsRDD[11] at flatMap at cmd15.sc:8
res15_3: Array[String] = Array("hola", "mundo", "spark", "es", "rápido")

2.3 filter — quedarse con los que cumplen una condición.

Devuelve un RDD con solo los elementos para los que la función devuelve true.

In [17]:
val numeros = sc.parallelize(1 to 10)
val pares = numeros.filter(n => n % 2 == 0)
pares.collect()
// Array(2, 4, 6, 8, 10)
println(pares.collect().mkString(", "))

2, 4, 6, 8, 10


numeros: org.apache.spark.rdd.RDD[Int] = ParallelCollectionRDD[14] at parallelize at cmd17.sc:1
pares: org.apache.spark.rdd.RDD[Int] = MapPartitionsRDD[15] at filter at cmd17.sc:2
res17_2: Array[Int] = Array(2, 4, 6, 8, 10)

In [18]:
val nombres = sc.parallelize(List("Ana", "Borja", "Carlos", "Beatriz", "Alberto"))
val conB = nombres.filter(n => n.startsWith("B"))
conB.collect()
// Array("Borja", "Beatriz")
println(conB.collect().mkString(", "))

Borja, Beatriz


nombres: org.apache.spark.rdd.RDD[String] = ParallelCollectionRDD[16] at parallelize at cmd18.sc:1
conB: org.apache.spark.rdd.RDD[String] = MapPartitionsRDD[17] at filter at cmd18.sc:2
res18_2: Array[String] = Array("Borja", "Beatriz")

3. Transformaciones de conjunto

3.1 union — combinar dos RDDs

Une dos RDDs del mismo tipo. No elimina duplicados.

In [19]:
val rdd1 = sc.parallelize(List(1, 2, 3))
val rdd2 = sc.parallelize(List(3, 4, 5))
rdd1.union(rdd2).collect()
// Array(1, 2, 3, 3, 4, 5)   ← el 3 aparece dos veces

rdd1: org.apache.spark.rdd.RDD[Int] = ParallelCollectionRDD[18] at parallelize at cmd19.sc:1
rdd2: org.apache.spark.rdd.RDD[Int] = ParallelCollectionRDD[19] at parallelize at cmd19.sc:2
res19_2: Array[Int] = Array(1, 2, 3, 3, 4, 5)

3.2 intersection — elementos comunes

Devuelve solo los elementos que están en ambos RDDs. Elimina duplicados.

In [20]:
val rdd1 = sc.parallelize(List(1, 2, 3, 4))
val rdd2 = sc.parallelize(List(3, 4, 5, 6))
rdd1.intersection(rdd2).collect()
// Array(3, 4)

rdd1: org.apache.spark.rdd.RDD[Int] = ParallelCollectionRDD[21] at parallelize at cmd20.sc:1
rdd2: org.apache.spark.rdd.RDD[Int] = ParallelCollectionRDD[22] at parallelize at cmd20.sc:2
res20_2: Array[Int] = Array(3, 4)

3.3 distinct — eliminar duplicados

In [ ]:
val conRepetidos = sc.parallelize(List(1, 2, 2, 3, 3, 3, 4))
conRepetidos.distinct().collect()
// Array(1, 2, 3, 4)   ← orden no garantizado

//estos no se ejecutan en VSC por el kernel Almond, entonces los realizo en Sparl-Shell donde corre perfectamente
//adjunto el archivo.scala en el github como ejercici.shufle14

java.lang.IllegalArgumentException: Unable to create serializer "com.esotericsoftware.kryo.serializers.FieldSerializer" for class: java.nio.HeapByteBuffer

4. Transformaciones clave-valor (Pair RDDs)

Cuando los elementos del RDD son tuplas (clave, valor), se desbloquean operaciones 

especiales muy útiles para análisis de datos.

4.1 groupByKey — agrupar valores por clave

Agrupa todos los valores que comparten la misma clave en una lista.

In [ ]:
val ventas = sc.parallelize(List(
  ("norte", 100), ("sur", 200), ("norte", 150), ("sur", 50), ("este", 300)
))
val agrupado = ventas.groupByKey()
agrupado.collect()
// Array(("norte", [100,150]), ("sur", [200,50]), ("este", [300]))

//Resuelto efectivamente en el archivo .scala con spark-shell

4.2 reduceByKey — reducir valores por clave

Combina los valores de la misma clave aplicando una función de dos parámetros. Es mucho 

más eficiente que groupByKey porque precombina localmente antes de enviar por la red.

In [ ]:
val ventas = sc.parallelize(List(
  ("norte", 100), ("sur", 200), ("norte", 150), ("sur", 50), ("este", 300)
))
val totalPorZona = ventas.reduceByKey((a, b) => a + b)
totalPorZona.collect()
// Array(("norte", 250), ("sur", 250), ("este", 300))
//spark-shell

4.3 sortBy y sortByKey — ordenar

In [ ]:
val numeros = sc.parallelize(List(5, 3, 1, 4, 2))
numeros.sortBy(n => n).collect()
// Array(1, 2, 3, 4, 5)

numeros.sortBy(n => n, ascending = false).collect()
// Array(5, 4, 3, 2, 1)
//spark-shell

In [ ]:
val pares = sc.parallelize(List(("c", 3), ("a", 1), ("b", 2)))
pares.sortByKey().collect()
// Array(("a",1), ("b",2), ("c",3))

pares.sortByKey(ascending = false).collect()
// Array(("c",3), ("b",2), ("a",1))
//spark-shell

4.4 join — combinar dos Pair RDDs por clave

Equivalente al JOIN de SQL. Une dos Pair RDDs por la clave común.

In [29]:
val empleados = sc.parallelize(List(
  (1, "Ana"), (2, "Borja"), (3, "Carlos")
))
val salarios = sc.parallelize(List(
  (1, 2500.0), (2, 3100.0), (4, 2800.0)  // clave 3 no existe, clave 4 no existe en empleados
))

val resultado = empleados.join(salarios)
resultado.collect()
// Array((1,("Ana",2500.0)), (2,("Borja",3100.0)))
// ← solo las claves que están en AMBOS RDDs (inner join)
println(resultado.collect().mkString(", "))

 Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp_06 - Ma Imp

(1,(Ana,2500.0)), (2,(Borja,3100.0))


empleados: org.apache.spark.rdd.RDD[(Int, String)] = ParallelCollectionRDD[60] at parallelize at cmd29.sc:1
salarios: org.apache.spark.rdd.RDD[(Int, Double)] = ParallelCollectionRDD[61] at parallelize at cmd29.sc:4
resultado: org.apache.spark.rdd.RDD[(Int, (String, Double))] = MapPartitionsRDD[64] at join at cmd29.sc:8
res29_3: Array[(Int, (String, Double))] = Array(
  (1, ("Ana", 2500.0)),
  (2, ("Borja", 3100.0))
)

Para incluir también los que no tienen pareja:

In [32]:
// leftOuterJoin: todos los de la izquierda, con o sin pareja
val izqJoin = empleados.leftOuterJoin(salarios).collect()
// Array((1,("Ana",Some(2500.0))), (2,("Borja",Some(3100.0))), (3,("Carlos",None)))
println(izqJoin.mkString(", "))

(1,(Ana,Some(2500.0))), (2,(Borja,Some(3100.0))), (3,(Carlos,None))


izqJoin: Array[(Int, (String, Option[Double]))] = Array(
  (1, ("Ana", Some(value = 2500.0))),
  (2, ("Borja", Some(value = 3100.0))),
  (3, ("Carlos", None))
)

RDD de entrada
    │
    ├─ map(f)           → RDD del mismo tamaño, cada elem transformado
    ├─ flatMap(f)       → RDD potencialmente más grande, aplanado
    ├─ filter(f)        → RDD más pequeño, solo los que cumplen condición
    ├─ distinct()       → RDD sin duplicados
    ├─ union(rdd2)      → RDD más grande (todos los elem de ambos)
    ├─ intersection(r2) → RDD más pequeño (solo los comunes)
    │
    └─ [Pair RDD (clave,valor)]
         ├─ groupByKey()    → (clave, Iterable[valor])
         ├─ reduceByKey(f)  → (clave, valor_reducido)
         ├─ sortByKey()     → Pair RDD ordenado por clave
         └─ join(rdd2)      → (clave, (v1, v2))

Ejercicios

Tienes una lista de temperaturas en Celsius. Conviértelas a Fahrenheit con la fórmula F = C * 9/5 + 32.

In [34]:
// CELDA — Ejercicio 1
val celsius = sc.parallelize(List(0.0, 20.0, 37.0, 100.0, -10.0))

val fahrenheit = celsius.map(c => c * 9.0 / 5.0 + 32.0)

println(fahrenheit.collect().mkString(", "))

32.0, 68.0, 98.6, 212.0, 14.0


celsius: org.apache.spark.rdd.RDD[Double] = ParallelCollectionRDD[79] at parallelize at cmd34.sc:2
fahrenheit: org.apache.spark.rdd.RDD[Double] = MapPartitionsRDD[80] at map at cmd34.sc:4

Ejercicio 2 — filter: filtrar productos
Tienes una lista de productos con precio. Filtra los que cuestan más de 50€.

In [ ]:
// CELDA — Ejercicio 2
val productos = sc.parallelize(List(
  ("Teclado", 35.99),
  ("Monitor", 199.99),
  ("Ratón", 22.50),
  ("Auriculares", 89.00),
  ("Alfombrilla", 12.00),
  ("Webcam", 65.00)
))

val caros = productos
  .filter { case (nombre, precio) => precio > 50.0 }
  .sortBy { case (nombre, precio) => -precio }  // ordenar de mayor a menor precio


println(caros.collect().mkString(", "))

(Monitor,199.99), (Auriculares,89.0), (Webcam,65.0)


productos: org.apache.spark.rdd.RDD[(String, Double)] = ParallelCollectionRDD[88] at parallelize at cmd36.sc:2
caros: org.apache.spark.rdd.RDD[(String, Double)] = MapPartitionsRDD[94] at sortBy at cmd36.sc:13

 Ejercicio 3 — flatMap: contar palabras
Dada una lista de frases, cuenta cuántas palabras tiene en total. Usa flatMap para obtener todas las palabras en un único RDD plano y luego count() para contar.

In [37]:
// CELDA — Ejercicio 3
val frases = sc.parallelize(List(
  "Apache Spark es un motor de procesamiento rápido",
  "Scala es el lenguaje principal de Spark",
  "Los RDDs son la base de Spark"
))

// flatMap divide cada frase por espacios y aplana el resultado
val palabras = frases.flatMap(f => f.split(" "))

println(s"Total de palabras: ${palabras.count()}")

Total de palabras: 22


frases: org.apache.spark.rdd.RDD[String] = ParallelCollectionRDD[95] at parallelize at cmd37.sc:2
palabras: org.apache.spark.rdd.RDD[String] = MapPartitionsRDD[96] at flatMap at cmd37.sc:9

Ejercicio 4 — distinct + union: operaciones de conjunto
Tienes los asistentes de dos eventos. Encuentra quién asistió a alguno de los dos (unión sin repetidos).

In [ ]:
// CELDA — Ejercicio 4
val eventoA = sc.parallelize(List("Ana", "Borja", "Carmen", "David"))
val eventoB = sc.parallelize(List("Carmen", "David", "Elena", "Fran"))

val todosAsistentes = eventoA.union(eventoB).distinct()

println("Asistentes a algún evento (sin repetidos):")
todosAsistentes.collect().sorted.foreach(println)



Ejercicio 5 — intersection: asistentes a ambos eventos
Usando los mismos datos de los dos eventos, encuentra quién asistió a los dos eventos.

⚠️ Importante: define los RDDs dentro de esta misma celda. Aunque los hayas creado en el ejercicio anterior, volver a definirlos aquí garantiza que el código funcione correctamente aunque ejecutes las celdas en distinto orden.

In [41]:
// CELDA — Ejercicio 5
// Redefinimos los RDDs para no depender de celdas anteriores
val eventoA2 = sc.parallelize(List("Ana", "Borja", "Carmen", "David"))
val eventoB2 = sc.parallelize(List("Carmen", "David", "Elena", "Fran"))

val ambosEventos = eventoA2.intersection(eventoB2)

println("Asistieron a AMBOS eventos:")
ambosEventos.collect().sorted.foreach(println)

Asistieron a AMBOS eventos:
Carmen
David


eventoA2: org.apache.spark.rdd.RDD[String] = ParallelCollectionRDD[115] at parallelize at cmd41.sc:3
eventoB2: org.apache.spark.rdd.RDD[String] = ParallelCollectionRDD[116] at parallelize at cmd41.sc:4
ambosEventos: org.apache.spark.rdd.RDD[String] = MapPartitionsRDD[122] at intersection at cmd41.sc:6

Ejercicio 6 — reduceByKey: ventas por departamento
Suma el total de ventas por departamento.

In [ ]:
// CELDA — Ejercicio 6
val ventas = sc.parallelize(List(
  ("Electrónica", 1200),
  ("Ropa", 340),
  ("Electrónica", 890),
  ("Hogar", 560),
  ("Ropa", 120),
  ("Electrónica", 450),
  ("Hogar", 230)
))

val totalPorDepto = ventas.reduceByKey((a, b) => a + b)

("Ventas totales por departamento:")
totalPorDepto.collect().sortBy(_._1).foreach {
  case (depto, total) => println(s"  $depto: ${total}€")
}

ventas: org.apache.spark.rdd.RDD[(String, Int)] = ParallelCollectionRDD[127] at parallelize at cmd44.sc:2
totalPorDepto: org.apache.spark.rdd.RDD[(String, Int)] = ShuffledRDD[128] at reduceByKey at cmd44.sc:12

 Ejercicio 7 — sortBy: ranking de ventas
Usando el resultado del ejercicio anterior, muestra los departamentos ordenados de mayor a menor venta con su posición en el ranking.

In [ ]:
// CELDA — Ejercicio 7
// sortBy(_._2, ascending = false) ordena por el segundo campo (ventas) de mayor a menor
val ranking = totalPorDepto.sortBy(_._2, ascending = false).collect()

println("Ranking de departamentos (mayor a menor venta):")
ranking.zipWithIndex.foreach { case ((depto, total), i) =>
  println(s"  ${i + 1}. $depto — ${total}€")
}

Ejercicio 8 — join: cruzar clientes con pedidos
Tienes dos RDDs: clientes y sus pedidos. Cruza ambos para mostrar el nombre del cliente junto con el importe de cada pedido.

In [46]:
// CELDA — Ejercicio 8
val clientes = sc.parallelize(List(
  (101, "Laura Sánchez"),
  (102, "Miguel Torres"),
  (103, "Patricia Ruiz"),
  (104, "Andrés López")
))

val pedidos = sc.parallelize(List(
  (101, 250.0),
  (102, 89.5),
  (101, 430.0),  // Laura tiene dos pedidos
  (103, 175.0),
  (105, 310.0)   // cliente 105 no existe en clientes
))

val resultado = clientes.join(pedidos)

println("Cliente → Pedido:")
resultado.collect().sortBy(_._1).foreach {
  case (id, (nombre, importe)) =>
    println(f"  [ID $id] $nombre → $importe%.2f€")
}

Cliente → Pedido:
  [ID 101] Laura Sánchez → 250,00€
  [ID 101] Laura Sánchez → 430,00€
  [ID 102] Miguel Torres → 89,50€
  [ID 103] Patricia Ruiz → 175,00€


clientes: org.apache.spark.rdd.RDD[(Int, String)] = ParallelCollectionRDD[132] at parallelize at cmd46.sc:2
pedidos: org.apache.spark.rdd.RDD[(Int, Double)] = ParallelCollectionRDD[133] at parallelize at cmd46.sc:9
resultado: org.apache.spark.rdd.RDD[(Int, (String, Double))] = MapPartitionsRDD[136] at join at cmd46.sc:17

 Ejercicio 9 — Pipeline completo: WordCount con ranking
El ejercicio clásico del Big Data: contar las palabras de un texto y mostrar las 5 más frecuentes. Fíjate cómo se encadenan 4 transformaciones en un solo pipeline.

In [ ]:
// CELDA — Ejercicio 9
val texto = sc.parallelize(List(
  "spark es rápido spark es potente",
  "scala es el lenguaje de spark",
  "spark procesa datos a gran velocidad",
  "los datos son el petróleo de scala",
  "spark scala big data de datos procesamiento"
))

val conteo = texto
  .flatMap(linea => linea.split(" "))  // paso 1: separar en palabras
  .map(palabra => (palabra, 1))         // paso 2: cada palabra vale 1
  .reduceByKey((a, b) => a + b)         // paso 3: sumar por palabra
  .sortBy(_._2, ascending = false)      // paso 4: ordenar de mayor a menor

println("Top 5 palabras más frecuentes:")
conteo.take(5).foreach { case (palabra, n) =>
  println(s"  '$palabra' → $n veces")
}

Ejercicio 10 — Análisis de logs de acceso
Tienes un RDD que simula líneas de un log web. Calcula el número de peticiones por código de estado HTTP.

In [50]:
// CELDA — Ejercicio 10
val logs = sc.parallelize(List(
  "192.168.1.1 GET /index.html 200",
  "10.0.0.2 POST /login 200",
  "192.168.1.3 GET /datos 404",
  "10.0.0.5 GET /index.html 200",
  "192.168.1.1 GET /admin 403",
  "10.0.0.8 GET /datos 404",
  "192.168.1.9 POST /login 500",
  "10.0.0.2 GET /index.html 200",
  "192.168.1.3 GET /admin 403",
  "10.0.0.5 GET /login 200"
))

// Extraer el código de estado (último campo) y contar
val peticionesPorCodigo = logs
  .map(linea => linea.split(" ").last)     // extraer el código
  .map(codigo => (codigo, 1))
  .reduceByKey((a, b) => a + b)
  .sortByKey()

println("Peticiones por código HTTP:")
peticionesPorCodigo.collect().foreach { case (codigo, total) =>
  val texto = if (total == 1) "petición" else "peticiones"
  println(s"  HTTP $codigo → $total $texto")
}

2026-04-23T08:51:53.179831900Z scala-kernel-interpreter-1 ERROR An exception occurred processing Appender console org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:164)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:133)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:124)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:88)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:714)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:672)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:648)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:584)
	at org.apache.logging.log4j.

java.lang.IllegalArgumentException: Unable to create serializer "com.esotericsoftware.kryo.serializers.FieldSerializer" for class: java.nio.HeapByteBuffer

Caso de Estudio Propuesto 1: MercaData S.L.
🏢 Contexto de la empresa
MercaData S.L. es una cadena de supermercados con tiendas en cuatro provincias españolas: Madrid, Barcelona, Valencia y Sevilla. Su departamento de tecnología ha decidido migrar el análisis de ventas a Apache Spark para poder procesar el volumen creciente de datos de forma distribuida.

Te han contratado como analista de datos para realizar el primer análisis real sobre los datos de la última semana. Los datos llegan en bruto como colecciones de cadenas de texto, tal y como los exporta su sistema de caja antiguo.

📂 Los datos
El sistema de caja exporta cada transacción como una línea de texto con este formato:

In [ ]:
"ID_TIENDA|PROVINCIA|PRODUCTO|CATEGORIA|IMPORTE|EMPLEADO_ID"

In [5]:
val empleados = sc.parallelize(List(
  ("E03", "Carmen Vidal"),
  ("E04", "Luis Herrero"),
  ("E07", "Marta Soler"),
  ("E08", "Diego Fuentes"),
  ("E11", "Ana Romero"),
  ("E12", "Pablo Leal"),
  ("E15", "Rosa Cano"),
  ("E16", "Javier Mora")
))

empleados: org.apache.spark.rdd.RDD[(String, String)] = ParallelCollectionRDD[1] at parallelize at cmd5.sc:1

Preguntas de negocio
Pregunta 1 — ¿Cuánto ha facturado cada provincia?
El director quiere saber el importe total vendido en cada provincia, ordenado de mayor a menor.

Pista: cada línea tiene la provincia en la posición 1 y el importe en la posición 4 (recuerda que split("\\|") divide por el carácter | y que toDouble convierte texto a número).

In [ ]:
val transacciones = sc.parallelize(List(
  "T01|Madrid|Leche Entera|Lácteos|1.20|E03",
  "T02|Barcelona|Pan de Molde|Panadería|1.85|E07",
  "T03|Valencia|Leche Entera|Lácteos|1.20|E11",
  "T04|Madrid|Zumo de Naranja|Bebidas|2.40|E03",
  "T05|Sevilla|Yogur Natural|Lácteos|0.75|E15",
  "T06|Barcelona|Agua Mineral|Bebidas|0.60|E07",
  "T07|Madrid|Cerveza Rubia|Bebidas|1.10|E04",
  "T08|Valencia|Pan de Molde|Panadería|1.85|E12",
  "T09|Sevilla|Leche Entera|Lácteos|1.20|E15",
  "T10|Madrid|Yogur Natural|Lácteos|0.75|E03",
  "T11|Barcelona|Zumo de Naranja|Bebidas|2.40|E08",
  "T12|Valencia|Cerveza Rubia|Bebidas|1.10|E11",
  "T13|Sevilla|Agua Mineral|Bebidas|0.60|E16",
  "T14|Madrid|Leche Entera|Lácteos|1.20|E04",
  "T15|Barcelona|Pan de Molde|Panadería|1.85|E07",
  "T16|Valencia|Yogur Natural|Lácteos|0.75|E12",
  "T17|Sevilla|Zumo de Naranja|Bebidas|2.40|E15",
  "T18|Madrid|Agua Mineral|Bebidas|0.60|E03",
  "T19|Barcelona|Leche Entera|Lácteos|1.20|E08",
  "T20|Valencia|Pan de Molde|Panadería|1.85|E11",
  "T21|Sevilla|Cerveza Rubia|Bebidas|1.10|E16",
  "T22|Madrid|Zumo de Naranja|Bebidas|2.40|E04",
  "T23|Barcelona|Yogur Natural|Lácteos|0.75|E07",
  "T24|Valencia|Leche Entera|Lácteos|1.20|E12",
  "T25|Sevilla|Pan de Molde|Panadería|1.85|E15"
))

val facturacionProvincia = (
  transacciones
    .map(linea => {
      val partes = linea.split("\\|")
      val provincia = partes(1)
      val importe = partes(4).toDouble
      (provincia, importe)
    })
    .reduceByKey(_ + _)
    .sortBy(_._2, ascending = false)
)

println("Facturación por provincia (mayor a menor):")

facturacionProvincia.collect().zipWithIndex.foreach {
  case ((provincia, total), i) =>
    println(s"${i + 1}. $provincia -> ${"%.2f".format(total)}€")
}

Pregunta 2 — ¿Qué categorías de producto se venden en Madrid?
El jefe de compras quiere saber qué categorías están presentes en las tiendas de Madrid, sin repetidos.

Pista: filtra primero por provincia, luego extrae la categoría y elimina duplicados.

In [ ]:
// ================= PREGUNTA 2 =================

val categoriasMadrid = (
  transacciones
    .map(linea => linea.split("\\|"))
    .filter(partes => partes(1) == "Madrid")
    .map(partes => partes(3))
    .distinct()
    .sortBy(categoria => categoria)
)

println("Categorías en Madrid:")
categoriasMadrid.collect().foreach(println)

Pregunta 3 — ¿Cuántas transacciones ha gestionado cada empleado?
RRHH necesita saber cuántas ventas ha procesado cada empleado esta semana, mostrando su nombre completo (no solo el ID).

Pista: necesitas dos pasos. Primero cuenta las transacciones por ID de empleado con reduceByKey. Luego cruza ese resultado con el RDD empleados usando join para sustituir el ID por el nombre.

In [ ]:
// ================= PREGUNTA 3 =================
// ¿Cuántas transacciones ha gestionado cada empleado?

// RDD de empleados
val empleados = sc.parallelize(List(
  ("E03", "Carmen Vidal"),
  ("E04", "Luis Herrero"),
  ("E07", "Marta Soler"),
  ("E08", "Diego Fuentes"),
  ("E11", "Ana Romero"),
  ("E12", "Pablo Leal"),
  ("E15", "Rosa Cano"),
  ("E16", "Javier Mora")
))

val transaccionesPorEmpleado = (
  transacciones
    .map(linea => {
      val partes = linea.split("\\|")
      val empleadoId = partes(5)
      (empleadoId, 1)
    })
    .reduceByKey(_ + _)
)

val resultado = (
  empleados
    .join(transaccionesPorEmpleado)
    .map {
      case (id, (nombre, total)) => (nombre, total)
    }
    .sortBy(_._1)
)

println("Transacciones por empleado:")

resultado.collect().foreach {
  case (nombre, total) =>
    val texto = if (total == 1) "transacción" else "transacciones"
    println(s"$nombre -> $total $texto")
}

In [ ]:
// ================= PREGUNTA 4 =================
// ¿Qué productos se venden tanto en Madrid como en Barcelona?

val datos = transacciones.map(_.split("\\|"))

val productosMadrid = (
  datos
    .filter(partes => partes(1) == "Madrid")
    .map(partes => partes(2))
    .distinct()
)

val productosBarcelona = (
  datos
    .filter(partes => partes(1) == "Barcelona")
    .map(partes => partes(2))
    .distinct()
)

val productosComunes = (
  productosMadrid
    .intersection(productosBarcelona)
    .sortBy(producto => producto)
)

println("Productos en Madrid y Barcelona:")
productosComunes.collect().foreach(println)

In [ ]:
// ================= PREGUNTA 5 =================
// ¿Cuál es la facturación total por categoría?

val facturacionCategoria = (
  transacciones
    .map(linea => {
      val partes = linea.split("\\|")
      val categoria = partes(3)
      val importe = partes(4).toDouble
      (categoria, importe)
    })
    .reduceByKey(_ + _)
    .sortBy(_._2, ascending = false)
)

println("Facturación por categoría:")

facturacionCategoria.collect().foreach {
  case (categoria, total) =>
    println(s"$categoria -> ${"%.2f".format(total)}€")
}

In [ ]:
val catalogoProductos = (
  transacciones
    .map(linea => linea.split("\\|")(2))
    .distinct()
    .sortBy(producto => producto)
)

println("Catálogo de productos:")
catalogoProductos.collect().foreach(println)